# AADS-ULoRA Colab: Testing and Validation

This notebook evaluates the trained models on test data and validates performance across all phases.

## What this notebook does:
1. **Load trained models** - Load all phase adapters
2. **Evaluate on test set** - Comprehensive performance metrics
3. **OOD detection validation** - Test anomaly detection
4. **Cross-domain validation** - Test generalization
5. **Generate predictions** - Save results for analysis

---
**Expected Runtime:** 30-60 minutes
**Required:** Trained models from all phases

---

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
import json
import logging
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def gate_check(step_id: str, check_name: str, condition: bool, expected: str, actual: str):
    status = "PASS" if condition else "FAIL"
    print(f"[{step_id}] {status} :: {check_name} | expected={expected} | actual={actual}")
    if not condition:
        raise RuntimeError(f"Gate failed at {step_id}::{check_name}")

# Add workspace to path
workspace_dir = Path('/content/aads_ulora')
sys.path.insert(0, str(workspace_dir))
gate_check('VALIDATION_SETUP', 'workspace_exists', workspace_dir.exists(), 'existing workspace directory', str(workspace_dir))

# Load configuration
config_path = workspace_dir / 'config' / 'colab.json'
gate_check('VALIDATION_SETUP', 'config_exists', config_path.exists(), 'colab.json exists', str(config_path))
with open(config_path, 'r') as f:
    config = json.load(f)

logger.info(f"✅ Configuration loaded from: {config_path}")

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Setup complete")

## 2. Import Dependencies

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

# Import models
from src.training.colab_phase1_training import ColabPhase1Trainer
from src.training.colab_phase2_sd_lora import ColabPhase2Trainer
from src.training.colab_phase3_conec_lora import ColabPhase3Trainer, CoNeCConfig
from src.utils.model_utils import extract_pooled_output

# Import data utilities
from src.dataset.colab_datasets import ColabCropDataset, ColabDomainShiftDataset
from src.dataset.colab_dataloader import ColabDataLoader

# Import evaluation metrics
from src.evaluation.metrics import (
    compute_accuracy,
    compute_precision_recall_f1,
    compute_ood_metrics,
    compute_confidence_intervals
)

# Import manifest validation helpers
from src.core.artifact_manifest import validate_manifest_artifacts

gate_check('VALIDATION_IMPORTS', 'validation_imports', True, 'all validation imports succeed', 'imports completed')

# Check CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

## 3. Load Test Dataset

In [ ]:
# Data paths
data_dir = workspace_dir / 'data'
metadata_path = data_dir / 'dataset_metadata.json'

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Metadata:")
print(json.dumps(metadata, indent=2))

# Create test dataset
image_size = config['data']['image_size']
normalize_mean = config['data']['normalize_mean']
normalize_std = config['data']['normalize_std']

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

test_path = data_dir / 'plantvillage' / 'test'

if not test_path.exists():
    print(f"⚠️  Test set not found at {test_path}")
    print("Using validation set for testing...")
    test_path = data_dir / 'plantvillage' / 'val'

test_dataset = ColabCropDataset(test_path, transform=test_transform)

test_loader = ColabDataLoader(
    test_dataset,
    batch_size=config['training']['phase3']['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✅ Test dataset loaded:")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Classes: {len(test_dataset.classes)}")
print(f"  Class names: {test_dataset.classes}")

## 4. Load Trained Models

In [ ]:
# Model paths
models_dir = workspace_dir / 'models'
phase1_path = models_dir / 'phase1_dora_adapter'
phase2_path = models_dir / 'phase2_sd_lora_adapter'
phase3_path = models_dir / 'phase3_conec_lora_adapter'

print("Checking for trained models...")

# Check Phase 1
gate_check('ARTIFACT_WRITE', 'phase1_adapter_dir', phase1_path.exists(), 'phase1 directory exists', str(phase1_path))
print(f"✅ Phase 1 adapter found: {phase1_path}")

# Check Phase 2
gate_check('ARTIFACT_WRITE', 'phase2_adapter_dir', phase2_path.exists(), 'phase2 directory exists', str(phase2_path))
print(f"✅ Phase 2 adapter found: {phase2_path}")

# Check Phase 3
gate_check('ARTIFACT_WRITE', 'phase3_adapter_dir', phase3_path.exists(), 'phase3 directory exists', str(phase3_path))
print(f"✅ Phase 3 adapter found: {phase3_path}")

# Manifest integrity checks
phase_requirements = {
    phase1_path: ['adapter_dir', 'classifier'],
    phase2_path: ['adapter_dir', 'classifier'],
    phase3_path: ['adapter_dir', 'classifier', 'prototypes', 'final_checkpoint'],
}

for phase_path, required_keys in phase_requirements.items():
    manifest_path = phase_path / 'manifest.json'
    gate_check('OUTPUT_SUITE_COMPLETE', f'{phase_path.name}_manifest_exists', manifest_path.exists(), 'manifest.json exists', str(manifest_path))
    validate_manifest_artifacts(manifest_path, required_keys)
    print(f"✅ Manifest validated: {manifest_path.name} for {phase_path.name}")

## 5. Phase 1 Model Evaluation

In [ ]:
# Load Phase 1 trainer
phase1_trainer = ColabPhase1Trainer(
    model_name=config['training']['phase1']['model_name'],
    num_classes=len(test_dataset.classes),
    device='cuda'
)

# Load checkpoint
phase1_checkpoint = phase1_path / 'checkpoint_epoch_10.pth'  # Adjust based on actual saved checkpoint
if phase1_checkpoint.exists():
    phase1_trainer.load_checkpoint(str(phase1_checkpoint))
    print(f"✅ Phase 1 model loaded from {phase1_checkpoint}")
else:
    # Try to find any checkpoint
    checkpoints = list(phase1_path.glob('checkpoint_epoch_*.pth'))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.stem.split('_')[-1]))
        phase1_trainer.load_checkpoint(str(latest_checkpoint))
        print(f"✅ Phase 1 model loaded from {latest_checkpoint}")
    else:
        print(f"❌ No checkpoints found in {phase1_path}")
        phase1_trainer = None

# Evaluate Phase 1
if phase1_trainer:
    print("\n" + "="*60)
    print("📊 Phase 1 Evaluation")
    print("="*60)
    
    phase1_trainer.model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = phase1_trainer.model(images)
            if hasattr(outputs, 'logits'):
                logits = outputs.logits
            else:
                logits = outputs
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Compute metrics
    phase1_accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Phase 1 Test Accuracy: {phase1_accuracy:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=test_dataset.classes,
                yticklabels=test_dataset.classes)
    plt.title('Phase 1 Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
else:
    print("❌ Skipping Phase 1 evaluation - model not available")

## 6. Phase 2 Model Evaluation

In [ ]:
# Load Phase 2 trainer
phase2_trainer = ColabPhase2Trainer(
    adapter_path=str(phase2_path),
    device='cuda'
)

# Load checkpoint
phase2_checkpoint = phase2_path / 'checkpoint_epoch_5.pth'  # Adjust based on actual saved checkpoint
if phase2_checkpoint.exists():
    phase2_trainer.load_checkpoint(str(phase2_checkpoint))
    print(f"✅ Phase 2 model loaded from {phase2_checkpoint}")
else:
    checkpoints = list(phase2_path.glob('checkpoint_epoch_*.pth'))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.stem.split('_')[-1]))
        phase2_trainer.load_checkpoint(str(latest_checkpoint))
        print(f"✅ Phase 2 model loaded from {latest_checkpoint}")
    else:
        print(f"❌ No checkpoints found in {phase2_path}")
        phase2_trainer = None

# Evaluate Phase 2
if phase2_trainer:
    print("\n" + "="*60)
    print("📊 Phase 2 Evaluation")
    print("="*60)
    
    phase2_trainer.model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = phase2_trainer.model(images)
            if hasattr(outputs, 'logits'):
                logits = outputs.logits
            else:
                logits = outputs
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    phase2_accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Phase 2 Test Accuracy: {phase2_accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))
else:
    print("❌ Skipping Phase 2 evaluation - model not available")

## 7. Phase 3 Model Evaluation (CoNeC-LoRA)

In [ ]:
# Load Phase 3 trainer
phase3_config = config['training']['phase3']
conec_config = CoNeCConfig(
    lora_r=phase3_config['lora_r'],
    lora_alpha=phase3_config['lora_alpha'],
    learning_rate=phase3_config['learning_rate'],
    batch_size=phase3_config['batch_size'],
    device='cuda',
    temperature=phase3_config['conec']['temperature'],
    prototype_dim=phase3_config['conec']['prototype_dim'],
    num_prototypes=phase3_config['conec']['num_prototypes'],
    contrastive_weight=phase3_config['conec']['contrastive_weight'],
    orthogonal_weight=phase3_config['conec']['orthogonal_weight']
)

phase3_trainer = ColabPhase3Trainer(
    config=conec_config,
    checkpoint_dir=str(phase3_path)
)

# Load checkpoint
phase3_checkpoint = phase3_path / 'checkpoint_epoch_10.pth'  # Adjust based on actual saved checkpoint
if phase3_checkpoint.exists():
    phase3_trainer.load_checkpoint(str(phase3_checkpoint))
    print(f"✅ Phase 3 model loaded from {phase3_checkpoint}")
else:
    checkpoints = list(phase3_path.glob('checkpoint_epoch_*.pth'))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.stem.split('_')[-1]))
        phase3_trainer.load_checkpoint(str(latest_checkpoint))
        print(f"✅ Phase 3 model loaded from {latest_checkpoint}")
    else:
        print(f"❌ No checkpoints found in {phase3_path}")
        phase3_trainer = None

# Evaluate Phase 3
if phase3_trainer:
    print("\n" + "="*60)
    print("📊 Phase 3 Evaluation")
    print("="*60)
    
    val_metrics = phase3_trainer.validate(test_loader)
    
    print(f"Test Loss: {val_metrics['loss']:.4f}")
    print(f"Test Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Contrastive Loss: {val_metrics.get('contrastive_loss', 0):.4f}")
    print(f"Orthogonal Loss: {val_metrics.get('orthogonal_loss', 0):.4f}")
    
    # OOD detection evaluation
    print("\n🔍 OOD Detection Evaluation:")
    ood_metrics_list = []
    with torch.no_grad():
        for batch in test_loader:
            images = batch['images'].to(phase3_trainer.device)
            labels = batch['labels'].to(phase3_trainer.device)
            
            pooled = extract_pooled_output(phase3_trainer.model, images)
            ood_metrics = phase3_trainer._perform_ood_detection(pooled, labels)
            ood_metrics_list.append(ood_metrics)
    
    if ood_metrics_list:
        avg_prototype_dist = np.mean([m['prototype_distances'].mean().item() for m in ood_metrics_list])
        print(f"  Average prototype distance: {avg_prototype_dist:.4f}")
        print(f"  OOD rate (prototype-based): {np.mean([m['prototype_anomaly'].mean().item() for m in ood_metrics_list]):.2%}")
        if 'mahalanobis_anomaly' in ood_metrics_list[0]:
            print(f"  OOD rate (Mahalanobis): {np.mean([m['mahalanobis_anomaly'].mean().item() for m in ood_metrics_list]):.2%}")
else:
    print("❌ Skipping Phase 3 evaluation - model not available")

## 8. Model Comparison

In [ ]:
# Compare all phases
print("\n" + "="*60)
print("📈 Model Comparison")
print("="*60)

comparison = {
    'Phase': ['Phase 1 (DoRA)', 'Phase 2 (SD-LoRA)', 'Phase 3 (CoNeC-LoRA)'],
    'Test Accuracy': [
        phase1_accuracy if 'phase1_accuracy' in locals() else None,
        phase2_accuracy if 'phase2_accuracy' in locals() else None,
        val_metrics['accuracy'] if 'val_metrics' in locals() else None
    ]
}

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

# Save comparison
results_dir = workspace_dir / 'outputs'
results_dir.mkdir(exist_ok=True)
df.to_csv(results_dir / 'model_comparison.csv', index=False)
print(f"\n✅ Comparison saved to: {results_dir / 'model_comparison.csv'}")

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(df['Phase'], df['Test Accuracy'], color=['#3498db', '#2ecc71', '#e74c3c'])
ax.set_ylabel('Test Accuracy')
ax.set_title('Model Performance Comparison')
ax.set_ylim(0, 1)

# Add value labels
for bar in bars:
    height = bar.get_height()
    if not np.isnan(height):
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 9. Generate Predictions on Test Set

In [ ]:
# Generate predictions using best model (Phase 3)
if phase3_trainer:
    print("\n" + "="*60)
    print("🔮 Generating Predictions")
    print("="*60)
    
    phase3_trainer.model.eval()
    predictions = []
    labels_list = []
    confidence_scores = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Predicting'):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            
            pooled = extract_pooled_output(phase3_trainer.model, images)
            logits = phase3_trainer.classifier(pooled)
            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            confidence_scores.extend(confidences.cpu().numpy())
    
    # Create results DataFrame
    results_df = pd.DataFrame({
        'true_label': labels_list,
        'predicted_label': predictions,
        'true_class': [test_dataset.classes[l] for l in labels_list],
        'predicted_class': [test_dataset.classes[p] for p in predictions],
        'confidence': confidence_scores,
        'correct': np.array(labels_list) == np.array(predictions)
    })
    
    # Save predictions
    predictions_path = results_dir / 'predictions.csv'
    results_df.to_csv(predictions_path, index=False)
    print(f"✅ Predictions saved to: {predictions_path}")
    
    # Summary statistics
    accuracy = results_df['correct'].mean()
    avg_confidence = results_df['confidence'].mean()
    
    print(f"\nSummary:")
    print(f"  Total samples: {len(results_df)}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Average confidence: {avg_confidence:.4f}")
    print(f"  Correct predictions: {results_df['correct'].sum()}/{len(results_df)}")
    
    # Confidence distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(results_df['confidence'], bins=20, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Confidence Distribution')
    axes[0].axvline(avg_confidence, color='red', linestyle='--', label=f'Mean: {avg_confidence:.3f}')
    axes[0].legend()
    
    # Confidence by correctness
    correct_conf = results_df[results_df['correct']]['confidence']
    incorrect_conf = results_df[~results_df['correct']]['confidence']
    axes[1].boxplot([correct_conf, incorrect_conf], labels=['Correct', 'Incorrect'])
    axes[1].set_ylabel('Confidence')
    axes[1].set_title('Confidence by Prediction Correctness')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ Skipping predictions - Phase 3 model not available")

## 10. OOD Detection Validation

In [ ]:
# Validate OOD detection performance
if phase3_trainer:
    print("\n" + "="*60)
    print("🔍 OOD Detection Validation")
    print("="*60)
    
    # Collect features and labels
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Extracting features'):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            pooled = extract_pooled_output(phase3_trainer.model, images)
            all_features.append(pooled.cpu())
            all_labels.append(labels.cpu())
    
    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # Get prototypes
    prototypes = phase3_trainer.prototype_manager.get_prototypes()
    print(f"Prototype shape: {prototypes.shape}")
    
    # Compute distances to prototypes
    distances = torch.cdist(all_features, prototypes)
    min_distances, nearest_prototypes = distances.min(dim=1)
    
    # Analyze distance distribution
    print(f"\nDistance Statistics:")
    print(f"  Mean distance: {min_distances.mean():.4f}")
    print(f"  Std distance: {min_distances.std():.4f}")
    print(f"  Min distance: {min_distances.min():.4f}")
    print(f"  Max distance: {min_distances.max():.4f}")
    
    # Plot distance distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(min_distances.numpy(), bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Distance to Nearest Prototype')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Prototype Distance Distribution')
    axes[0].axvline(min_distances.mean(), color='red', linestyle='--', label=f'Mean: {min_distances.mean():.3f}')
    axes[0].legend()
    
    # Box plot by class
    distance_by_class = [min_distances[all_labels == i].numpy() for i in range(len(test_dataset.classes))]
    axes[1].boxplot(distance_by_class, labels=test_dataset.classes)
    axes[1].set_ylabel('Distance to Nearest Prototype')
    axes[1].set_title('Distance by Class')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Save OOD analysis
    ood_analysis = {
        'mean_distance': float(min_distances.mean()),
        'std_distance': float(min_distances.std()),
        'min_distance': float(min_distances.min()),
        'max_distance': float(min_distances.max()),
        'threshold_suggestions': {
            'mean_plus_2std': float(min_distances.mean() + 2 * min_distances.std()),
            'mean_plus_3std': float(min_distances.mean() + 3 * min_distances.std()),
            'percentile_95': float(np.percentile(min_distances.numpy(), 95))
        }
    }
    
    ood_analysis_path = results_dir / 'ood_analysis.json'
    with open(ood_analysis_path, 'w') as f:
        json.dump(ood_analysis, f, indent=2)
    print(f"\n✅ OOD analysis saved to: {ood_analysis_path}")
else:
    print("❌ Skipping OOD validation - Phase 3 model not available")

## 11. Test Summary Report

In [ ]:
# Generate comprehensive test summary
print("\n" + "="*60)
print("📋 TEST SUMMARY REPORT")
print("="*60)

summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'test_dataset': {
        'path': str(test_path),
        'num_samples': len(test_dataset),
        'num_classes': len(test_dataset.classes),
        'classes': test_dataset.classes
    },
    'models': {
        'phase1_available': phase1_trainer is not None,
        'phase2_available': phase2_trainer is not None,
        'phase3_available': phase3_trainer is not None
    },
    'artifacts': {},
    'results': {}
}

if 'phase_requirements' in locals():
    for phase_path, required_keys in phase_requirements.items():
        manifest_path = phase_path / 'manifest.json'
        summary['artifacts'][phase_path.name] = {
            'manifest_path': str(manifest_path),
            'manifest_exists': manifest_path.exists(),
            'required_keys': required_keys,
        }

if 'phase1_accuracy' in locals():
    summary['results']['phase1_accuracy'] = float(phase1_accuracy)
if 'phase2_accuracy' in locals():
    summary['results']['phase2_accuracy'] = float(phase2_accuracy)
if 'val_metrics' in locals():
    summary['results']['phase3_accuracy'] = float(val_metrics['accuracy'])
    summary['results']['phase3_loss'] = float(val_metrics['loss'])
    summary['results']['phase3_contrastive_loss'] = float(val_metrics.get('contrastive_loss', 0))
    summary['results']['phase3_orthogonal_loss'] = float(val_metrics.get('orthogonal_loss', 0))

# Print summary
print("\nDataset Information:")
print(f"  Test samples: {summary['test_dataset']['num_samples']}")
print(f"  Classes: {summary['test_dataset']['num_classes']}")
print(f"  Class names: {', '.join(summary['test_dataset']['classes'])}")

print("\nModel Availability:")
for phase, available in summary['models'].items():
    status = "✅ Available" if available else "❌ Not available"
    print(f"  {phase}: {status}")

if summary['artifacts']:
    print("\nArtifact Manifests:")
    for phase_name, artifact_info in summary['artifacts'].items():
        status = "✅ Manifest" if artifact_info['manifest_exists'] else "❌ Missing"
        print(f"  {phase_name}: {status}")

if summary['results']:
    print("\nTest Results:")
    for metric, value in summary['results'].items():
        print(f"  {metric}: {value:.4f}")

# Save summary
summary_path = results_dir / 'test_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✅ Summary saved to: {summary_path}")

print("\n" + "="*60)
print("✅ Testing and Validation Complete!")
print("="*60)

## 12. Interactive VLM Upload Test (Plant Type + Plant Part)

Use this section to upload diseased plant images repeatedly in Colab and verify strict VLM routing outputs:
- plant type
- plant part
- confidence

This section requires valid model IDs in `config/colab.json` under `router.vlm.model_ids`.

In [ ]:
# Interactive strict VLM test setup
import json
import sys
from pathlib import Path

# Prefer mounted Drive project root for persistence
project_root = Path('/content/drive/MyDrive/aads_ulora')
if not project_root.exists():
    project_root = Path('/content/aads_ulora')

if not project_root.exists():
    raise FileNotFoundError(
        'Project root not found. Expected /content/drive/MyDrive/aads_ulora or /content/aads_ulora'
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'config' / 'colab.json'
if not config_path.exists():
    raise FileNotFoundError(f'Config not found: {config_path}')

with open(config_path, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)

vlm_cfg = _cfg.get('router', {}).get('vlm', {})
crop_id = vlm_cfg.get('model_ids', {}).get('crop', '')
part_id = vlm_cfg.get('model_ids', {}).get('part', '')

if not crop_id or not part_id:
    raise ValueError(
        'Strict mode requires model IDs. Set router.vlm.model_ids.crop and router.vlm.model_ids.part in config/colab.json'
    )

from scripts.colab_interactive_vlm_test import run_interactive_vlm_test

print(f'✅ Project root: {project_root}')
print(f'✅ Config: {config_path}')
print(f'✅ Crop model ID: {crop_id}')
print(f'✅ Part model ID: {part_id}')
print('Ready for interactive uploads.')

In [ ]:
# Run interactive upload loop (re-run this cell any time)
run_interactive_vlm_test(config_path=str(config_path))